In [ ]:
import numpy as np
import numpy.typing as npt
from astropy.time import Time
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.controller_v2.tracker_controller import TrackerController
from sorts.controller_v2.fence_scan_controller_new import FenceScanController
from sorts.schedule_v2 import Schedule, ExperimentDetail
from sorts.scheduler_v2.priority_scheduling import _priority_scheduling_df

# import for plottings
import pandas as pd
from sorts import plots

In [ ]:
pd.set_option("display.expand_frame_repr", False)

In [ ]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z

# the first set of value used, not much use now; kept for ref
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# a 1 sec long period, the sbobj should be very close to right up ahead of eiscat3d tx-0 station
# start_time = Time("2025-01-01 04:04:00")
# end_time = Time("2025-01-01 04:04:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# an extended duration which expands around from the 1 sec period above
# the `control_slice_duration` is much longer than normal, practical radar `control_slice_duration`
# for easier debugging, inspection of scheduling/schedules
# start_time = Time("2025-01-01 02:45:00")
# end_time = Time("2025-01-01 06:15:00")
# control_slice_duration = np.timedelta64(int(60 * 1e6), "us")

# same as above, but use more realistic 10ms `control_slice_duration`
start_time = Time("2025-01-01 02:45:00")
end_time = Time("2025-01-01 06:15:00")
control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

eiscat3d = get_radar("eiscat3d", "stage1-array")

spobj = SpaceObject(
    SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)


exp_detail_0 = ExperimentDetail(
    id=0,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

exp_detail_1 = ExperimentDetail(
    id=1,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    start_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    end_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    control_slice_duration,
)
# time_arr = time_arr[::4] # TODO: remove; strided to bring up the effects of scheduling
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - epoch.to_value("datetime64").astype("datetime64[us]")  # type: ignore
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = spobj.get_state(dsec_arr)

trackerController = TrackerController(
    tx_station=eiscat3d.tx[0],
    rx_stations=[],
    time=time_arr,
    space_object_states=ecefs,
    exp_detail=exp_detail_0,
    min_elevation=10,
)

fenceScanController = FenceScanController(
    tx_station=eiscat3d.tx[0],
    rx_station=[],
    exp_datail=exp_detail_1,
    azimuth=90, # sweep from east to west
    min_elevation=30,
    pointings_per_cycle=40,
)

In [ ]:
ecefs.shape

In [ ]:
plots.ecef_states_positions_plot(ecefs)

In [ ]:
tracker_schs = trackerController.generate()
tracker_tx_sch_df = tracker_schs.tx_schedule.as_dataframe()
tracker_tx_sch_df

In [ ]:
# check schedule df memory usage (MB)
tracker_tx_sch_df.memory_usage().sum()/1e6

In [ ]:
fence_schs = fenceScanController.generate(start_time, end_time)
fence_tx_sch_df = fence_schs.tx_schedule.as_dataframe()
fence_tx_sch_df

In [ ]:
# check schedule df memory usage (MB)
fence_tx_sch_df.memory_usage().sum()/1e6

In [ ]:
tx_sch = tracker_schs.tx_schedule
plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [ ]:
tx_sch = fence_schs.tx_schedule
plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [ ]:
(tracker_schs.tx_schedule.meta, fence_schs.tx_schedule.meta)

In [ ]:
master_sch_meta = {0: exp_detail_0, 1: exp_detail_1}
master_sch_df = _priority_scheduling_df([tracker_schs.tx_schedule, fence_schs.tx_schedule])
master_sch_df

In [ ]:
master_sch_df[master_sch_df["exp_num"] == 1]

In [ ]:
master_sch = Schedule.from_dataframe(master_sch_df, master_sch_meta)
master_sch

In [ ]:
plots.schedule_plot(master_sch)